# 06 Transfer learning — InceptionV3

Zamrznuta ImageNet baza + nova dense glava, na istoj fiksiranoj podeli kao custom modeli.

Ulaz je **299×299** (nativno za InceptionV3), a feature-i zamrznute baze se **kesiraju**
jednom na disk, pa se glava trenira nad 2048-dim vektorima.

Kernel: **venv sa TensorFlow-om**.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path("..").resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import tensorflow as tf
print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## Brza provera lanca

Malo slika po klasi, par epoha — samo da se vidi da kesiranje, oblici i labele rade.
Pise u zaseban folder, da ne pokvari pravi eksperiment.

In [ ]:
!python src/transfer.py --exp-dir experiments/_smoke_transfer --limit 180 --epochs 3

## Kesiranje feature-a

Jednokratno, nad celim skupom. Na CPU-u traje dugo (18k slika kroz InceptionV3).

In [ ]:
!python src/transfer.py --stage cache

## Trening dense glave

Nad keširanim feature-ima — sekunde po epohi, pa je 100 epoha jeftino.
Cuva se najbolja po `val_accuracy`, a zatim se sklapa pun model slika → softmax.

In [ ]:
!python src/transfer.py --stage dense --epochs 100

In [ ]:
!python src/plot_curves.py --model transfer_dense

## Evaluacija na test skupu

Pokrece se **tek na kraju**, nad `best.keras`.

In [ ]:
!python src/evaluate.py --model transfer_dense --exp-dir experiments/transfer_dense

In [ ]:
import json
from IPython.display import Image, display

with open("experiments/transfer_dense/run_config.json") as f:
    print("run config:", json.load(f))

with open("experiments/transfer_dense/test_metrics.json") as f:
    print("test metrics:", json.load(f))

display(Image("reports/figures/transfer_dense_curves.png"))
display(Image("reports/figures/transfer_dense_confusion.png"))

## Poredjenje sa custom modelima

Slucajno pogadjanje za 18 klasa je **5.56%**. Referentne vrednosti iz rada:
VGG-lite ~31%, hibrid ~36%, transfer learning ~56-57%.

In [ ]:
import json
from pathlib import Path

import pandas as pd

rows = []
for name, exp in (
    ("VGG-lite", "experiments/custom_vgglite"),
    ("Hibrid", "experiments/custom_hybrid"),
    ("InceptionV3 (dense)", "experiments/transfer_dense"),
):
    metrics_path = Path(exp) / "test_metrics.json"
    if not metrics_path.is_file():
        continue
    with open(metrics_path) as f:
        m = json.load(f)
    rows.append({
        "model": name,
        "params": m.get("params"),
        "test accuracy": m.get("test_accuracy"),
        "macro F1": m.get("macro_f1"),
    })

pd.DataFrame(rows)